# Ingredientes


In [1]:
# @title Ruta de Google Drive
# @markdown Usada para validar si esta montada la unidad de Google Drive
mydrive_gdrive_path = "/content/drive/MyDrive";

In [2]:
# @title Rutas de carpetas
import os
ruta_base_tesis = os.path.join(mydrive_gdrive_path, "01_proyecto_grado_MIS")
ruta_imagenes = os.path.join(ruta_base_tesis, 'Version_2_ConADN/0_documentacion/01_proyecto/imagenes')

# Reseta para activar google drive

In [3]:
# @title Función para montar un drive de google

def montar_google_drive():

  from google.colab import drive
  drive.mount('/content/drive')

In [4]:
# @title Función para crear carpeta
import os

def crear_carpeta(carpeta):

  if(not os.path.exists(mydrive_gdrive_path)):
    montar_google_drive()

  os.makedirs(carpeta, exist_ok=True)

In [5]:
# @title Función para copiar archivos
import shutil
import os

def copiar_archivos(carpeta_origen, carpeta_destino):
  imagenes_totales=0
  imagenes_copiadas=0

  # Verifica si las carpetas existen
  if not os.path.isdir(carpeta_origen):
      print(f"La carpeta de origen '{carpeta_origen}' no existe.")
  elif not os.path.isdir(carpeta_destino):
      print(f"La carpeta de destino '{carpeta_destino}' no existe.")
  else:
      # Itera sobre los archivos en la carpeta de origen
      for nombre_archivo in os.listdir(carpeta_origen):
        imagenes_totales+=1
        if nombre_archivo.lower().endswith((".jpg", ".jpeg", ".png")):
          ruta_origen_completa = os.path.join(carpeta_origen, nombre_archivo)
          ruta_destino_completa = os.path.join(carpeta_destino, nombre_archivo)

          # Copia el archivo (puedes usar copy() o copy2())
          try:
              shutil.copy2(ruta_origen_completa, ruta_destino_completa)
              imagenes_copiadas+=1
              #print(f"'{nombre_archivo}' copiado a '{ruta_destino_completa}'")
          except Exception as e:
              print(f"Error al copiar '{nombre_archivo}': {e}")
  print(f"Imágenes copiadas {imagenes_copiadas} de  {imagenes_totales}.")

In [6]:
# @title Función para copiar archivo
import shutil
import os

def copiar_archivo(archivo_origen, archivo_destino, modificar=False):
  """
  Copia un archivo desde una ruta de origen a una ruta de destino.
  Valida la existencia del archivo de origen y crea la carpeta de destino si no existe.
  """

  # Verificar si el archivo de origen existe
  if os.path.exists(archivo_destino) and not modificar:
    print(f"Error: El archivo destino '{archivo_origen}' existe. No se puede modificar. Enviar variable modificar en True para forzar actualización.")
    return

  # Verificar si el archivo de origen existe
  if not os.path.exists(archivo_origen):
    print(f"Error: El archivo de origen '{archivo_origen}' no existe.")
    return

  # Obtener el directorio de destino
  directorio_destino = os.path.dirname(archivo_destino)

  # Crear el directorio de destino si no existe
  if not os.path.exists(directorio_destino):
    os.makedirs(directorio_destino, exist_ok=True)
    print(f"Carpeta de destino '{directorio_destino}' creada.")

  # Copiar el archivo
  try:
    shutil.copy2(archivo_origen, archivo_destino)
    print(f"Archivo '{archivo_origen}' copiado a '{archivo_destino}' exitosamente.")
  except Exception as e:
    print(f"Error al copiar el archivo '{archivo_origen}' a '{archivo_destino}': {e}")

In [7]:
# @title Función para eliminar archivo
import os

def eliminar_archivo(ruta_archivo):
  """
  Elimina un archivo dado su ruta.
  """
  try:
    if os.path.exists(ruta_archivo):
      os.remove(ruta_archivo)
      print(f"Archivo '{ruta_archivo}' eliminado exitosamente.")
    else:
      print(f"El archivo '{ruta_archivo}' no existe.")
  except Exception as e:
    print(f"Error al eliminar el archivo '{ruta_archivo}': {e}")

In [8]:
# @title Función para mover un archivo
import os

def mover_archivo(carpeta_origen, archivo_origen, carpeta_destino, archivo_destino):
  # Ensure the destination directory exists
  os.makedirs(carpeta_destino, exist_ok=True)

  # Move the file
  source_path = os.path.join(carpeta_origen, archivo_origen)
  destination_path = os.path.join(carpeta_destino, archivo_destino)

  try:
      os.rename(source_path, destination_path)
      print(f"File moved successfully from {source_path} to {destination_path}")
  except FileNotFoundError:
      print(f"Error: The source file was not found at {source_path}")
  except FileExistsError:
      print(f"Error: A file with the same name already exists at {destination_path}")
  except Exception as e:
      print(f"An error occurred: {e}")

# Reseta para aumentar imagenes

In [9]:
# @title Clase para aumentar
import numpy as np
from PIL import Image

class FiltrarAugmentation():
  def __init__(self, transform):
    self.transform = transform

  def procesar(self, imagen):
    img_np = np.array(imagen)
    augmented = self.transform(image=img_np)
    imagen_filtrada = augmented['image']
    resultado_pil = Image.fromarray(imagen_filtrada)
    return resultado_pil

In [10]:
# @title Función aumentar imagenes
import uuid
import albumentations as A

def aumentar_imagen(ruta, archivo):
  if not archivo.lower().endswith((".jpg", ".jpeg", ".png")):
    print(f"❌ Formato de imagen no valido: {archivo}")
    return

  transform_horizontal_flip = A.Compose([A.HorizontalFlip(p=1)])
  filtro_horizontal_flip = FiltrarAugmentation(transform_horizontal_flip)

  transform_translate = A.Compose([A.Affine(
          translate_percent={"x": 0.1, "y": 0.1},   # reemplaza shift_limit
        )])
  filtro_translate = FiltrarAugmentation(transform_translate)

  transform_scale = A.Compose([A.Affine(scale=(1-0.1,1+0.1))])
  filtro_scale = FiltrarAugmentation(transform_scale)

  transform_rotate = A.Compose([A.Affine(rotate=(-1*90, 90))])
  filtro_rotate = FiltrarAugmentation(transform_rotate)

  transform_brightness = A.Compose([A.RandomBrightnessContrast(brightness_limit=0.20, p=1)])
  filtro_brightness = FiltrarAugmentation(transform_brightness)

  transform_contrast = A.Compose([A.RandomBrightnessContrast(contrast_limit=0.20, p=1)])
  filtro_contrast = FiltrarAugmentation(transform_contrast)

  transform_hue = A.Compose([A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=8, val_shift_limit=8, p=1)])
  filtro_hue = FiltrarAugmentation(transform_hue)

  archivo_sin_extension=archivo.lower().replace(".jpg", "").replace(".jpeg", "").replace(".png", "")
  ruta = os.path.join(ruta, archivo)

  try:
    try:
      imagen = Image.open(ruta)
      # Convert image to RGB if it's in RGBA mode
      if imagen.mode == 'RGBA':
        imagen = imagen.convert('RGB')
    except FileNotFoundError:
      print(f"❌ Archivo no encontrado: {ruta}")
      return

    # Generate a random UUID object
    archivo_uuid = uuid.uuid4()

    # image_aug = filtro_horizontal_flip.procesar(imagen)
    image_aug = filtro_hue.procesar(imagen)
    nombre_imagen_destino_aug = "aug_hue_"+archivo_sin_extension+"_"+str(archivo_uuid)+"_"+archivo.replace(archivo_sin_extension,"")
    destino = os.path.join(ruta_imagenes, nombre_imagen_destino_aug)
    image_aug.save(destino)


  except Exception as e:
    print(f"❌ Error al revisar y aumentar la imagen: {e}, {ruta}")

  return

  # transform = A.Compose([
  #     A.HorizontalFlip(p=self.horizontal_flip),                # volteo horizontal
  #     A.Affine(
  #       translate_percent={"x": self.shift_limit, "y": self.shift_limit},   # reemplaza shift_limit
  #       scale=(1 - self.scale_limit, 1 + self.scale_limit),                    # reemplaza scale_limit
  #       rotate=(-1*self.rotate_limit, self.rotate_limit),                            # reemplaza rotate_limit
  #       p=self.shift_scale_rotate_p
  #     )
  # ])

  # transform = A.Compose([
  #   A.RandomBrightnessContrast(
  #       brightness_limit=0.09,   # pequeñas variaciones ±8%
  #       contrast_limit=0.09,     # pequeñas variaciones ±9%
  #       p=self.random_brightness_contrast_p
  #   ),
  #   A.HueSaturationValue(
  #       hue_shift_limit=8,       # máximo ±7° → casi imperceptible
  #       sat_shift_limit=9,      # pequeñas variaciones
  #       val_shift_limit=9,      # evita cambios extremos
  #       p=self.hue_saturation_value_p
  #   )
  # ])

# Mise en place

In [11]:
# @title Script para montar Google Drive
montar_google_drive()

Mounted at /content/drive


# Cocinar

In [14]:
# @title Script para aumentar una imagen

ruta = ruta_imagenes
imagen = 'imagen_original.jpg'
aumentar_imagen(ruta, imagen)